# FIT · Tableau export

Turns the raw data into files for a Tableau Public dashboard. This is a side product of the pipeline, not a FIT layer, and it only uses games that passed every check in `01_pull_and_clean.ipynb`.

Output in `data/tableau/`:

- `shots.csv`: one row per shot, with court location and zone.
- `lineups.csv`: one row per five-man lineup per team, without garbage time.
- `court.png`: half court drawing to put behind the shot chart.

In [ ]:
import os

import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib.patches import Arc, Circle, Rectangle

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

In [ ]:
DATA_DIR = 'data'
OUT_DIR = os.path.join(DATA_DIR, 'tableau')

os.makedirs(OUT_DIR, exist_ok = True)

games = pd.read_csv(os.path.join(DATA_DIR, 'games.csv'), dtype = {'GAME_ID': str})
stints = pd.read_csv(os.path.join(DATA_DIR, 'stints.csv'), dtype = {'GAME_ID': str})
players = pd.read_csv(os.path.join(DATA_DIR, 'players.csv'))

good_games = games.loc[games['GAME_ID'].isin(stints['GAME_ID'])]

names = players.set_index('PERSON_ID')['PLAYER']

print(f'Games: {len(good_games)}, from {good_games["GAME_DATE"].min()} to {good_games["GAME_DATE"].max()}')

## Shots

The coordinates are in tenths of a foot with the hoop at (0, 0). The baseline is at y = -52.5 and half court is at y = 417.5.

`shotDistance` from NBA.com can't be trusted: many corner threes are listed at 0 feet. So distance is calculated from the coordinates. Whether a shot is a 2 or a 3 comes from `shotValue`, which is what was actually scored.

In [ ]:
pbp = pd.concat([pd.read_csv(os.path.join(DATA_DIR, 'raw', 'pbp', f'{game_id}.csv')).assign(GAME_ID = game_id)
                 for game_id in good_games['GAME_ID']])

shots = pbp.loc[pbp['isFieldGoal'] == 1].copy()

shots['SHOT_DIST'] = (np.hypot(shots['xLegacy'], shots['yLegacy']) / 10).round(1)
shots['MADE'] = (shots['shotResult'] == 'Made').astype(int)
shots['POINTS'] = shots['MADE'] * shots['shotValue']


def get_zone(x, y, value, dist):

    if y >= 417.5:
        return 'Backcourt'

    if value == 3:
        return 'Corner 3' if y <= 87.5 else 'Above the Break 3'

    if dist <= 4:
        return 'Restricted Area'

    return 'Paint (Non-RA)' if abs(x) < 80 and y < 137.5 else 'Mid-Range'


shots['ZONE'] = [get_zone(x, y, v, d) for x, y, v, d in zip(shots['xLegacy'], shots['yLegacy'], shots['shotValue'], shots['SHOT_DIST'])]

shots = shots.merge(good_games, on = 'GAME_ID')

shots['IS_HOME'] = shots['teamId'] == shots['HOME_ID']
shots['OPPONENT'] = np.where(shots['IS_HOME'], shots['AWAY'], shots['HOME'])
shots['PLAYER'] = shots['personId'].map(names).fillna(shots['playerName'])

shots = shots.rename(columns = {'period': 'PERIOD', 'teamTricode': 'TEAM', 'personId': 'PLAYER_ID', 'xLegacy': 'X', 'yLegacy': 'Y',
                                'shotValue': 'SHOT_VALUE', 'subType': 'SHOT_TYPE'})

# stored on each shot because Tableau's AVG of a FIXED field counts each zone once, not each shot
shots['LEAGUE_ZONE_PPS'] = shots.groupby('ZONE')['POINTS'].transform('mean')

shots = shots[['GAME_ID', 'GAME_DATE', 'PERIOD', 'TEAM', 'OPPONENT', 'IS_HOME', 'PLAYER_ID', 'PLAYER', 'X', 'Y', 'SHOT_DIST',
               'ZONE', 'SHOT_TYPE', 'SHOT_VALUE', 'MADE', 'POINTS', 'LEAGUE_ZONE_PPS']]

shots.to_csv(os.path.join(OUT_DIR, 'shots.csv'), index = False)

print(f'Shots: {len(shots):,}')

shots.groupby('ZONE').agg(FGA = ('MADE', 'size'), FG_PCT = ('MADE', 'mean'), PPS = ('POINTS', 'mean'), DIST = ('SHOT_DIST', 'mean')).round(3).sort_values('FGA', ascending = False)

Check: shot points plus made free throws have to add up to the official final score in every game.

In [ ]:
ft_made = pbp.loc[(pbp['actionType'] == 'Free Throw') & ~pbp['description'].str.contains('MISS', na = False)].groupby(['GAME_ID', 'teamTricode']).size()

scored = shots.groupby(['GAME_ID', 'TEAM'])['POINTS'].sum().add(ft_made.rename_axis(['GAME_ID', 'TEAM']), fill_value = 0)

official = pd.concat([good_games.set_index(['GAME_ID', 'HOME'])['HOME_PTS'].rename_axis(['GAME_ID', 'TEAM']),
                      good_games.set_index(['GAME_ID', 'AWAY'])['AWAY_PTS'].rename_axis(['GAME_ID', 'TEAM'])])

diff = scored.sub(official, fill_value = 0)

print(f'Team games: {len(diff)}, matching the official score: {(diff == 0).sum()}')

Check: few 2s should sit past the three-point line, and the league numbers should look like the NBA.

In [ ]:
print(f'2s from 24+ feet: {((shots["SHOT_VALUE"] == 2) & (shots["SHOT_DIST"] >= 24)).sum()}')
print(f'3s from under 22 feet: {((shots["SHOT_VALUE"] == 3) & (shots["SHOT_DIST"] < 22)).sum()}')

print(f'FG%: {shots["MADE"].mean():.3f}, 3P%: {shots.loc[shots["SHOT_VALUE"] == 3, "MADE"].mean():.3f}, share of shots that are 3s: {(shots["SHOT_VALUE"] == 3).mean():.3f}')

## Lineups

Each stint counts once for each team. Points and possessions are kept as raw totals, so Tableau can calculate the ratings correctly at any level. Averaging ratings that were already calculated would give a 2-minute lineup the same weight as a 200-minute one.

Garbage time is left out, the same as in the RAPM baseline.

In [ ]:
home_cols = ['H1', 'H2', 'H3', 'H4', 'H5']
away_cols = ['A1', 'A2', 'A3', 'A4', 'A5']

stints = stints.merge(good_games[['GAME_ID', 'HOME', 'AWAY']], on = 'GAME_ID')


def one_side(stints, cols, team, pts_for, pts_against, poss_for, poss_against):

    side = stints[['GAME_ID', 'SECONDS', 'GARBAGE']].copy()

    side['TEAM'] = stints[team]
    side['LINEUP_ID'] = stints[cols].astype(str).agg('-'.join, axis = 1)
    side['PTS_FOR'], side['PTS_AGAINST'] = stints[pts_for], stints[pts_against]
    side['POSS_FOR'], side['POSS_AGAINST'] = stints[poss_for], stints[poss_against]

    return side


sides = pd.concat([one_side(stints, home_cols, 'HOME', 'PTS_HOME', 'PTS_AWAY', 'POSS_HOME', 'POSS_AWAY'),
                   one_side(stints, away_cols, 'AWAY', 'PTS_AWAY', 'PTS_HOME', 'POSS_AWAY', 'POSS_HOME')])

played = sides.loc[~sides['GARBAGE']]

lineups = played.groupby(['TEAM', 'LINEUP_ID']).agg(GAMES = ('GAME_ID', 'nunique'), SECONDS = ('SECONDS', 'sum'),
                                                    PTS_FOR = ('PTS_FOR', 'sum'), PTS_AGAINST = ('PTS_AGAINST', 'sum'),
                                                    POSS_FOR = ('POSS_FOR', 'sum'), POSS_AGAINST = ('POSS_AGAINST', 'sum')).reset_index()

lineups['MINUTES'] = (lineups['SECONDS'] / 60).round(2)
lineups['PLAYERS'] = lineups['LINEUP_ID'].apply(lambda ids: ', '.join(sorted(names.get(int(i), i) for i in ids.split('-'))))

lineups = lineups[['TEAM', 'LINEUP_ID', 'PLAYERS', 'GAMES', 'MINUTES', 'PTS_FOR', 'PTS_AGAINST', 'POSS_FOR', 'POSS_AGAINST']]

lineups.to_csv(os.path.join(OUT_DIR, 'lineups.csv'), index = False)

print(f'Lineups: {len(lineups):,}')
print(f'Lineups with 100+ minutes: {(lineups["MINUTES"] >= 100).sum()}')
print(f'Lineups with under 10 minutes: {(lineups["MINUTES"] < 10).sum()}')

Check: with garbage time included, each team's stint points have to match the official final score, and the league has to look like the league.

In [ ]:
stint_points = sides.groupby(['GAME_ID', 'TEAM'])['PTS_FOR'].sum()

print(f'Team games matching the official score: {(stint_points.sub(official, fill_value = 0) == 0).sum()} of {len(official)}')

print(f'League points per 100 possessions: {100 * lineups["PTS_FOR"].sum() / lineups["POSS_FOR"].sum():.1f}')
print(f'Average minutes per team per game: {sides["SECONDS"].sum() / 60 / len(official):.1f}')

big = lineups.loc[lineups['MINUTES'] >= 100].copy()
big['NET'] = 100 * (big['PTS_FOR'] / big['POSS_FOR'] - big['PTS_AGAINST'] / big['POSS_AGAINST'])

big.sort_values('MINUTES', ascending = False).head(10)[['TEAM', 'PLAYERS', 'GAMES', 'MINUTES', 'NET']].round(1)

## Court image

Drawn with the same coordinates as the shots, so it lines up in Tableau as long as the image limits are set to x from -250 to 250 and y from -52.5 to 417.5.

In [ ]:
LINE = '#8a8f98'

fig = plt.figure(figsize = (10, 9.4), dpi = 150)
ax = fig.add_axes([0, 0, 1, 1])

ax.set_xlim(-250, 250)
ax.set_ylim(-52.5, 417.5)
ax.axis('off')

court = [Rectangle((-250, -52.5), 500, 470),
         Circle((0, 0), 7.5),
         Rectangle((-80, -52.5), 160, 190),
         Rectangle((-60, -52.5), 120, 190),
         Arc((0, 137.5), 120, 120, theta1 = 0, theta2 = 180),
         Arc((0, 137.5), 120, 120, theta1 = 180, theta2 = 360, linestyle = '--'),
         Arc((0, 0), 80, 80, theta1 = 0, theta2 = 180),
         Arc((0, 0), 475, 475, theta1 = 22, theta2 = 158),
         Arc((0, 417.5), 120, 120, theta1 = 180, theta2 = 360)]

for part in court:
    part.set(edgecolor = LINE, linewidth = 1.5, fill = False)
    ax.add_patch(part)

# the backboard and the straight part of the three-point line
ax.plot([-30, 30], [-7.5, -7.5], color = LINE, linewidth = 1.5)
ax.plot([-220, -220], [-52.5, 89], color = LINE, linewidth = 1.5)
ax.plot([220, 220], [-52.5, 89], color = LINE, linewidth = 1.5)

fig.savefig(os.path.join(OUT_DIR, 'court.png'), facecolor = 'white')
plt.show()